# M7 — Luồng A: Train + tuning + lưu model

Đọc `listings_features/sale` (M6, macro đã as-of join) → **chia train/test** → thử 3 regressor (LinearRegression, RandomForest, GBT) với `CrossValidator` + `ParamGridBuilder` → **ablation vĩ mô** (có vs không macro) → lưu **PipelineModel đã fit** có version + bảng chỉ số.

**Quyết định split (user 2026-09-17):** dùng **random 80/20 (seed=42)**, KHÔNG chia theo năm 2025/2026.
> Lý do: data 2142/2156 dòng dồn Dec-2025, chỉ 14 dòng 2026 → chia theo năm cho test n=14 (vô nghĩa thống kê). Random split cho test ~431 dòng, chỉ số ổn định. Đánh đổi: lệch chủ ý chia-theo-thời-gian của spec M7 — chấp nhận vì toàn bộ data gần như cùng một cửa sổ thời gian nên leakage thời gian không đáng kể.

**Target:** `price_per_m2` (triệu/m²).

**Model lưu = FULL PipelineModel** (feature stages fit trên train + regressor) → M8 nạp 1 artifact, `transform` tin mới (đã as-of join macro) → prediction. Tránh lệch train/serve.

## 1. Config + Java + imports

In [1]:
import os, json, datetime
if os.path.basename(os.getcwd()) == "ml":
    os.chdir("..")
os.environ.setdefault(
    "JAVA_HOME",
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
)

from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    SQLTransformer, StringIndexer, OneHotEncoder,
    VectorAssembler, StandardScaler,
)
from pyspark.ml.regression import (
    LinearRegression, RandomForestRegressor, GBTRegressor,
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd

FEAT_IN   = "data/lake/listings_features/sale"   # input (M6)
LABEL     = "price_per_m2"
VERSION   = datetime.date.today().isoformat()    # vd 2026-09-17
MODEL_DIR = f"models/v{VERSION}"
SEED      = 42

## 2. Spark + đọc feature dataset + chia train/test (random 80/20)

In [2]:
spark = (
    SparkSession.builder.appName("M7-train")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

feat = spark.read.parquet(FEAT_IN)
train, test = feat.randomSplit([0.8, 0.2], seed=SEED)
train.cache(); test.cache()
n_train, n_test = train.count(), test.count()
print(f"tổng {feat.count()} | train {n_train} | test {n_test}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 19:41:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/09/17 19:41:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


tổng 2156 | train 1775 | test 381


## 3. Feature stages (tái dựng M6) — biến thể có macro / không macro

Ablation vĩ mô = build 2 bộ đặc trưng giống hệt nhau, chỉ khác việc có đưa `gold_usd/usdvnd/vnindex` vào `VectorAssembler` hay không. Mọi biến đổi per-row nằm trong pipeline → M8 tái dựng y hệt.

In [3]:
# SQLTransformer y hệt M6 (cột phái sinh; tên cột chotot có dấu cách -> backtick)
DERIVE_SQL = """
    SELECT *,
        log(`Dien tich`)                              AS log_area,
        year(`Ngay dang`)                             AS year,
        month(`Ngay dang`)                            AS month,
        quarter(`Ngay dang`)                          AS quarter,
        dayofweek(`Ngay dang`)                        AS dayofweek,
        6371 * 2 * asin(sqrt(
            power(sin(radians(`Vi do` - 10.7769) / 2), 2) +
            cos(radians(10.7769)) * cos(radians(`Vi do`)) *
            power(sin(radians(`Kinh do` - 106.7009) / 2), 2)
        ))                                            AS dist_center,
        coalesce(`Loai BDS`, 'UNKNOWN')               AS loai_bds_s
    FROM __THIS__
"""

NUM_BASE = [
    "log_area", "Phong ngu", "Nha ve sinh", "So tang", "rank_quan", "dist_center",
    "year", "month", "quarter", "dayofweek",
]
MACRO = ["gold_usd", "usdvnd", "vnindex"]

def feature_stages(use_macro: bool):
    """5 stage đặc trưng M6; toggle 3 cột macro trong assembler."""
    derive = SQLTransformer(statement=DERIVE_SQL)
    idx = StringIndexer(inputCol="loai_bds_s", outputCol="loai_idx", handleInvalid="keep")
    ohe = OneHotEncoder(inputCol="loai_idx", outputCol="loai_ohe", handleInvalid="keep")
    num = NUM_BASE + (MACRO if use_macro else [])
    asm = VectorAssembler(inputCols=num + ["loai_ohe"], outputCol="features_raw",
                          handleInvalid="error")
    scaler = StandardScaler(inputCol="features_raw", outputCol="features",
                            withStd=True, withMean=False)
    return [derive, idx, ohe, asm, scaler]

## 4. Regressor + lưới tham số (tuning)

`CrossValidator` 3-fold chọn tham số theo RMSE trên train. Lưới nhỏ gọn (đủ minh họa "cách điều chỉnh tham số" cho báo cáo, không đốt compute).

In [4]:
def build_cases():
    """(tên, estimator, paramGrid-builder-fn theo estimator)."""
    def lr_grid(m):
        return (ParamGridBuilder()
                .addGrid(m.regParam, [0.0, 0.1])
                .addGrid(m.elasticNetParam, [0.0, 0.5]).build())
    def rf_grid(m):
        return (ParamGridBuilder()
                .addGrid(m.numTrees, [50, 100])
                .addGrid(m.maxDepth, [5, 10]).build())
    def gbt_grid(m):
        return (ParamGridBuilder()
                .addGrid(m.maxDepth, [3, 5])
                .addGrid(m.maxIter, [30, 50]).build())
    return [
        ("LinearRegression", LinearRegression(featuresCol="features", labelCol=LABEL), lr_grid),
        ("RandomForest",     RandomForestRegressor(featuresCol="features", labelCol=LABEL, seed=SEED), rf_grid),
        ("GBT",              GBTRegressor(featuresCol="features", labelCol=LABEL, seed=SEED), gbt_grid),
    ]

ev_rmse = RegressionEvaluator(labelCol=LABEL, predictionCol="prediction", metricName="rmse")
ev_mae  = RegressionEvaluator(labelCol=LABEL, predictionCol="prediction", metricName="mae")
ev_r2   = RegressionEvaluator(labelCol=LABEL, predictionCol="prediction", metricName="r2")

## 5. Train + tune tất cả (model × có/không macro)

Mỗi ô: `CrossValidator` fit **toàn bộ pipeline** (feature stages fit CHỈ trên train fold → scaler/indexer không leakage) → best → đánh giá trên test.

In [5]:
results = []
best_models = {}   # key -> bestModel (full PipelineModel)

for use_macro in (True, False):
    tag = "macro" if use_macro else "no_macro"
    fstages = feature_stages(use_macro)
    for name, est, grid_fn in build_cases():
        pipe = Pipeline(stages=fstages + [est])
        cv = CrossValidator(
            estimator=pipe,
            estimatorParamMaps=grid_fn(est),
            evaluator=ev_rmse,
            numFolds=3, seed=SEED, parallelism=2,
        )
        cv_model = cv.fit(train)
        best = cv_model.bestModel
        pred = best.transform(test)
        row = {
            "model": name, "features": tag,
            "rmse": ev_rmse.evaluate(pred),
            "mae":  ev_mae.evaluate(pred),
            "r2":   ev_r2.evaluate(pred),
            "cv_rmse_train": min(cv_model.avgMetrics),
        }
        results.append(row)
        best_models[(name, tag)] = best
        print(f"[{tag:8}] {name:16} test RMSE={row['rmse']:.3f} MAE={row['mae']:.3f} R2={row['r2']:.3f}")

26/09/17 19:41:45 WARN Instrumentation: [c3729e56] regParam is zero, which might cause numerical instability and overfitting.
26/09/17 19:41:45 WARN Instrumentation: [7d1716e0] regParam is zero, which might cause numerical instability and overfitting.


26/09/17 19:41:46 WARN Instrumentation: [c3729e56] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/09/17 19:41:46 WARN Instrumentation: [7d1716e0] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


26/09/17 19:41:47 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/09/17 19:41:47 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/09/17 19:41:47 ERROR LBFGS: Failure again! Giving up and returning. Maybe the objective is just poorly behaved?
26/09/17 19:41:47 ERROR LBFGS: Failure again! Giving up and returning. Maybe the objective is just poorly behaved?


26/09/17 19:41:49 WARN Instrumentation: [fb3ac9ed] regParam is zero, which might cause numerical instability and overfitting.
26/09/17 19:41:49 WARN Instrumentation: [fea23a67] regParam is zero, which might cause numerical instability and overfitting.
26/09/17 19:41:49 WARN Instrumentation: [fb3ac9ed] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/09/17 19:41:49 WARN Instrumentation: [fea23a67] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/09/17 19:41:49 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/09/17 19:41:49 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/09/17 19:41:49 ERROR LBFGS: Failure again! Giving up and returning. Maybe the objective is just poorly behaved?
26/09/17 19:41:49 ERROR LBFGS: Failure again! Giving up and returning. Maybe the objective is just poorly

26/09/17 19:41:50 WARN Instrumentation: [1cc87a91] regParam is zero, which might cause numerical instability and overfitting.
26/09/17 19:41:50 WARN Instrumentation: [e61d6ec8] regParam is zero, which might cause numerical instability and overfitting.
26/09/17 19:41:50 WARN Instrumentation: [1cc87a91] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/09/17 19:41:50 WARN Instrumentation: [e61d6ec8] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/09/17 19:41:50 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/09/17 19:41:50 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/09/17 19:41:50 ERROR LBFGS: Failure again! Giving up and returning. Maybe the objective is just poorly behaved?
26/09/17 19:41:50 ERROR LBFGS: Failure again! Giving up and returning. Maybe the objective is just poorly

...

26/09/17 19:43:12 WARN DAGScheduler: Broadcasting large task binary with size 1308.8 KiB
26/09/17 19:43:13 WARN DAGScheduler: Broadcasting large task binary with size 2006.9 KiB


26/09/17 19:43:13 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB


26/09/17 19:43:13 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB


[no_macro] RandomForest     test RMSE=15.057 MAE=11.029 R2=0.372


[no_macro] GBT              test RMSE=16.300 MAE=11.374 R2=0.264


## 6. Bảng chỉ số + ablation vĩ mô

In [6]:
res = pd.DataFrame(results).sort_values(["features", "rmse"]).reset_index(drop=True)
print("=== Bảng chỉ số (test) — target price_per_m2 (triệu/m²) ===")
print(res.to_string(index=False))

print("\n=== Ablation vĩ mô (cùng model, macro − no_macro; RMSE âm = macro tốt hơn) ===")
piv = res.pivot(index="model", columns="features", values="rmse")
piv["delta_rmse"] = piv["macro"] - piv["no_macro"]
print(piv.to_string())
print("\nGhi chú: macro biến thiên nhỏ (data dồn ~11 ngày Dec-2025) → cải thiện dự kiến khiêm tốn; báo cáo trung thực theo số.")

=== Bảng chỉ số (test) — target price_per_m2 (triệu/m²) ===
           model features      rmse       mae       r2  cv_rmse_train
    RandomForest    macro 15.127579 11.107075 0.365901      13.488853
             GBT    macro 15.952112 11.321625 0.294893      14.266126
LinearRegression    macro 17.441587 13.763982 0.157072      16.227358
    RandomForest no_macro 15.057413 11.029024 0.371769      13.417634
             GBT no_macro 16.299521 11.374017 0.263847      14.199253
LinearRegression no_macro 17.496971 13.817806 0.151710      16.229634

=== Ablation vĩ mô (cùng model, macro − no_macro; RMSE âm = macro tốt hơn) ===
features              macro   no_macro  delta_rmse
model                                             
GBT               15.952112  16.299521   -0.347409
LinearRegression  17.441587  17.496971   -0.055384
RandomForest      15.127579  15.057413    0.070166

Ghi chú: macro biến thiên nhỏ (data dồn ~11 ngày Dec-2025) → cải thiện dự kiến khiêm tốn; báo cáo trung thực theo 

## 7. Chọn model tốt nhất (có macro) + lưu có version

Model sản phẩm dùng **có macro** (đúng kiến trúc Luồng B ghép vĩ mô). Chọn RMSE test nhỏ nhất trong nhóm macro. Lưu full `PipelineModel` + `metrics.json` (baseline RMSE cho drift check M8).

In [7]:
macro_rows = [r for r in results if r["features"] == "macro"]
best_row = min(macro_rows, key=lambda r: r["rmse"])
best_key = (best_row["model"], "macro")
best_model = best_models[best_key]

os.makedirs(MODEL_DIR, exist_ok=True)
best_model.write().overwrite().save(f"{MODEL_DIR}/model")

metrics = {
    "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "version": VERSION,
    "target": LABEL,
    "split": {"method": "random", "ratio": [0.8, 0.2], "seed": SEED,
              "n_train": n_train, "n_test": n_test},
    "results": results,
    "best": {"model": best_row["model"], "features": "macro",
             "rmse": best_row["rmse"], "mae": best_row["mae"], "r2": best_row["r2"]},
    "baseline_rmse": best_row["rmse"],   # M8 drift check so với số này
}
with open(f"{MODEL_DIR}/metrics.json", "w") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f"model tốt nhất (macro): {best_row['model']} | test RMSE={best_row['rmse']:.3f} R2={best_row['r2']:.3f}")
print(f"lưu -> {MODEL_DIR}/model")
print(f"metrics -> {MODEL_DIR}/metrics.json | baseline_rmse={metrics['baseline_rmse']:.3f}")

26/09/17 19:44:00 WARN TaskSetManager: Stage 10193 contains a task of very large size (1346 KiB). The maximum recommended task size is 1000 KiB.


model tốt nhất (macro): RandomForest | test RMSE=15.128 R2=0.366
lưu -> models/v2026-09-17/model
metrics -> models/v2026-09-17/metrics.json | baseline_rmse=15.128


## 8. Verify — nạp lại model + predict thử

In [8]:
from pyspark.ml import PipelineModel
reloaded = PipelineModel.load(f"{MODEL_DIR}/model")
sample = reloaded.transform(test).select(LABEL, "prediction").limit(5)
print("nạp lại OK. Mẫu (thực vs dự đoán, triệu/m²):")
sample.show(truncate=False)
print("=" * 55)
print("M7 XONG:")
print(f"  split      : random 80/20 seed={SEED} (train {n_train} / test {n_test})")
print(f"  best model : {best_row['model']} (macro) RMSE={best_row['rmse']:.3f} R2={best_row['r2']:.3f}")
print(f"  path       : {MODEL_DIR}/model")
spark.stop()

nạp lại OK. Mẫu (thực vs dự đoán, triệu/m²):
+-----------------+-----------------+
|price_per_m2     |prediction       |
+-----------------+-----------------+
|29.68421052631579|42.74506330983285|
|52.5             |64.75329175333403|
|39.09090909090909|42.05480674977476|
|100.0            |61.23492638526884|
|62.5             |71.13274928190977|
+-----------------+-----------------+

M7 XONG:
  split      : random 80/20 seed=42 (train 1775 / test 381)
  best model : RandomForest (macro) RMSE=15.128 R2=0.366
  path       : models/v2026-09-17/model
